# M5 PPE（安全帽偵測）訓練（Colab，GPU 版）

資料集：[Hard Hat Workers](https://doi.org/10.7910/DVN/7CBGOS)（Harvard Dataverse，CC0 1.0）。

**限制**：這個資料集只有 `helmet`（安全帽）／`head`（沒戴安全帽的頭）兩類，**沒有反光背心類別**——這是資料集本身的限制，不是漏做。查證紀錄見專案 `docs/licenses.md`。

使用前：**Runtime → Change runtime type → T4 GPU**。

In [ ]:
!pip install -q ultralytics==8.4.160
!apt-get -qq install -y unrar > /dev/null

## 1. 下載資料集與轉換腳本

In [ ]:
!mkdir -p /content/hardhat
!curl -sL -o /content/hardhat/Hardhat.rar "https://dataverse.harvard.edu/api/access/datafile/3344658"
!unrar x -o+ /content/hardhat/Hardhat.rar /content/hardhat/ > /dev/null
!git clone --depth 1 https://github.com/thothawei/vision-ai-demo.git /content/vision-ai-demo

In [ ]:
import sys, pathlib, importlib.util
sys.path.insert(0, "/content/vision-ai-demo")

spec = importlib.util.spec_from_file_location("prepare_hardhat", "/content/vision-ai-demo/scripts/prepare_hardhat.py")
prepare_hardhat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prepare_hardhat)

prepare_hardhat.RAW_DIR = pathlib.Path("/content/hardhat/Hardhat")
prepare_hardhat.OUT_DIR = pathlib.Path("/content/hardhat_yolo")
prepare_hardhat.main()

## 2. 訓練

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data="/content/hardhat_yolo/data.yaml",
    epochs=60,
    imgsz=640,
    batch=32,
    device=0,
    patience=15,
    project="/content/runs",
    name="ppe_yolo11n",
    seed=42,
)

In [ ]:
metrics = model.val(data="/content/hardhat_yolo/data.yaml")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 3. 下載權重，放回本機專案的 `models/ppe/ppe_yolo11n/weights/best.pt`

In [ ]:
from google.colab import files
files.download("/content/runs/ppe_yolo11n/weights/best.pt")